## LANL Comprehensive, Multi-Source Cyber-Security Events — Bronze ETL

**Source:** Los Alamos National Laboratory ([doi:10.17021/1179829](http://dx.doi.org/10.17021/1179829))  
**Coverage:** 58 consecutive days · 1.65B events · 12,425 users · 17,684 computers · 62,974 processes  
**License:** CC0

| Bronze Table | Source File | Description | Key Columns |
|---|---|---|---|
| `auth` | auth.txt (\~73 GB) | Windows auth events (desktop + AD) | time, source_user, destination_user, source/dest computer, auth_type, logon_type, success_failure |
| `dns` | dns.txt (\~813 MB) | DNS lookups from internal servers | time, source_computer, resolved_computer |
| `flows` | flows.txt (\~5.2 GB) | Network flows from routers | time, duration, source/dest computer+port, protocol, packet_count, byte_count |
| `proc` | proc.txt (\~15 GB) | Process start/stop on Windows hosts | time, user, computer, process_name, start_end |
| `redteam` | redteam.txt (\~23 KB) | Ground-truth compromise events | time, user, source_computer, destination_computer |

**Notes:** All identifiers are de-identified and unified across files (e.g. U1 is the same user everywhere). Time starts at epoch 1 with 1-second resolution. Missing values are `?`.  
**Citation:** A. D. Kent, *Cybersecurity Data Sources for Dynamic Network Research*, Dynamic Networks in Cybersecurity, 2015.

---

### Table of Contents

1. [Setup & Configuration](#1-setup--configuration)
2. [Bronze Table Ingestion](#2-bronze-table-ingestion) — raw CSV → Delta tables
3. [Validation](#3-validation) — row-count sanity checks

---

## 1. Setup & Configuration

Import libraries, define volume paths, and ensure the `workspace.bronze_lanl` schema exists.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType
from pyspark.sql.functions import current_timestamp, col

VOLUME_PATH = "/Volumes/workspace/default/lanl"
CATALOG = "workspace"
SCHEMA = "bronze_lanl"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✓ Schema {CATALOG}.{SCHEMA} ready")

## 2. Bronze Table Ingestion

Each cell below reads a raw CSV from the LANL Volume, applies a strict schema, appends ingestion metadata (`_ingested_at`, `_source_file`), and writes to a managed Delta table. No transformations — data lands as-is.

In [0]:
auth_schema = StructType([
    StructField("time", LongType()),
    StructField("source_user", StringType()),
    StructField("destination_user", StringType()),
    StructField("source_computer", StringType()),
    StructField("destination_computer", StringType()),
    StructField("auth_type", StringType()),
    StructField("logon_type", StringType()),
    StructField("auth_orientation", StringType()),
    StructField("success_failure", StringType())
])

df_auth = (spark.read
    .schema(auth_schema)
    .option("header", "false")
    .csv(f"{VOLUME_PATH}/auth.txt")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

df_auth.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.auth")
print(f"✓ auth -> {CATALOG}.{SCHEMA}.auth")

In [0]:
dns_schema = StructType([
    StructField("time", LongType()),
    StructField("source_computer", StringType()),
    StructField("resolved_computer", StringType())
])

df_dns = (spark.read
    .schema(dns_schema)
    .option("header", "false")
    .csv(f"{VOLUME_PATH}/dns.txt")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

df_dns.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dns")
print(f"✓ dns -> {CATALOG}.{SCHEMA}.dns")

In [0]:
flows_schema = StructType([
    StructField("time", LongType()),
    StructField("duration", IntegerType()),
    StructField("source_computer", StringType()),
    StructField("source_port", StringType()),
    StructField("destination_computer", StringType()),
    StructField("destination_port", StringType()),
    StructField("protocol", IntegerType()),
    StructField("packet_count", IntegerType()),
    StructField("byte_count", LongType())
])

df_flows = (spark.read
    .schema(flows_schema)
    .option("header", "false")
    .csv(f"{VOLUME_PATH}/flows.txt")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

df_flows.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.flows")
print(f"✓ flows -> {CATALOG}.{SCHEMA}.flows")

In [0]:
proc_schema = StructType([
    StructField("time", LongType()),
    StructField("user", StringType()),
    StructField("computer", StringType()),
    StructField("process_name", StringType()),
    StructField("start_end", StringType())
])

df_proc = (spark.read
    .schema(proc_schema)
    .option("header", "false")
    .csv(f"{VOLUME_PATH}/proc.txt")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

df_proc.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.proc")
print(f"✓ proc -> {CATALOG}.{SCHEMA}.proc")

In [0]:
redteam_schema = StructType([
    StructField("time", LongType()),
    StructField("user", StringType()),
    StructField("source_computer", StringType()),
    StructField("destination_computer", StringType())
])

df_redteam = (spark.read
    .schema(redteam_schema)
    .option("header", "false")
    .csv(f"{VOLUME_PATH}/redteam.txt")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

df_redteam.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.redteam")
print(f"✓ redteam -> {CATALOG}.{SCHEMA}.redteam")

## 3. Validation

Quick row-count check across all bronze tables to confirm ingestion completed.

In [0]:
from pyspark.sql.functions import lit

tables = ["auth", "dns", "flows", "proc", "redteam"]
counts = []
for t in tables:
    n = spark.table(f"{CATALOG}.{SCHEMA}.{t}").count()
    counts.append((t, n))

df_counts = spark.createDataFrame(counts, ["table", "row_count"])
display(df_counts)